# Qwen3-VL OCR Engine - Google Colab

This notebook runs the Qwen3-VL-2B-Instruct model for OCR tasks on GPU.

## Features
- GPU-accelerated inference
- Structured JSON output with sections and key-value pairs
- Section-level bounding box visualization
- Interactive configuration widgets
- Support for multiple prompt profiles (free_ocr, markdown, form, table)
- Automatic bounding box coordinate conversion (normalized to pixels)


In [ ]:
# Install dependencies
!pip install -q transformers>=4.56.0 accelerate>=0.30.0 qwen-vl-utils[decord]==0.0.8
!pip install -q pydantic ipywidgets


In [ ]:
# Imports
import json
import tempfile
import time
from pathlib import Path
from typing import List, Optional

import torch
from PIL import Image, ImageDraw
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

import matplotlib.pyplot as plt
from google.colab import files
import ipywidgets as widgets
from IPython.display import display
from pydantic import BaseModel, Field, ValidationError


In [ ]:
# Load Qwen3-VL model on GPU
def load_qwen_model(model_id="Qwen/Qwen3-VL-2B-Instruct"):
    """Load Qwen3-VL model and processor on GPU."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch_dtype = torch.float16 if device == "cuda" else torch.float32
    
    print(f"Loading model on {device} with dtype {torch_dtype}...")
    print(f"Model: {model_id}")
    
    # Load processor
    processor = AutoProcessor.from_pretrained(
        model_id,
        trust_remote_code=True,
    )
    
    # Load model
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch_dtype,
        device_map="auto" if device == "cuda" else "cpu",
        trust_remote_code=True,
    )
    
    model = model.eval()
    
    print(f"Model loaded successfully on {device}")
    return model, processor, device

# Load the model
model, processor, device = load_qwen_model()


In [ ]:
# Pydantic Schema Definitions for Structured Output

class BoundingBox(BaseModel):
    """Bounding box coordinates."""
    x1: int = Field(..., description="Left x coordinate", ge=0)
    y1: int = Field(..., description="Top y coordinate", ge=0)
    x2: int = Field(..., description="Right x coordinate", ge=0)
    y2: int = Field(..., description="Bottom y coordinate", ge=0)
    
    def to_dict(self):
        return {"x1": self.x1, "y1": self.y1, "x2": self.x2, "y2": self.y2}

class KeyValuePair(BaseModel):
    """A key-value pair extracted from the document."""
    key: str = Field(..., description="Field name or label")
    value: str = Field(..., description="Extracted value")
    confidence: float = Field(default=1.0, ge=0.0, le=1.0, description="Confidence score")
    
    def to_dict(self):
        return {
            "key": self.key,
            "value": self.value,
            "confidence": self.confidence
        }

class Section(BaseModel):
    """A document section with bounding box and extracted data."""
    section_id: str = Field(..., description="Unique section identifier")
    section_type: str = Field(..., description="Section type (header, body, footer, table, etc.)")
    bbox: BoundingBox = Field(..., description="Bounding box coordinates")
    key_value_pairs: List[KeyValuePair] = Field(default_factory=list, description="Extracted key-value pairs")
    text: str = Field(..., description="Full text content of the section")
    
    def to_dict(self):
        return {
            "section_id": self.section_id,
            "section_type": self.section_type,
            "bbox": self.bbox.to_dict(),
            "key_value_pairs": [kv.to_dict() for kv in self.key_value_pairs],
            "text": self.text
        }

class DocumentMetadata(BaseModel):
    """Metadata about the document and processing."""
    document_type: str = Field(..., description="Type of document (invoice, form, letter, etc.)")
    engine: str = Field(default="Qwen3-VL", description="OCR engine used")
    model_variant: str = Field(default="2B-Instruct", description="Model variant")
    processing_time_ms: float = Field(..., description="Processing time in milliseconds")
    image_size: dict = Field(..., description="Image dimensions")
    
    def to_dict(self):
        return {
            "document_type": self.document_type,
            "engine": self.engine,
            "model_variant": self.model_variant,
            "processing_time_ms": self.processing_time_ms,
            "image_size": self.image_size
        }

class OCRResult(BaseModel):
    """Complete OCR result with structured sections."""
    metadata: DocumentMetadata
    sections: List[Section] = Field(default_factory=list, description="Document sections")
    
    def to_dict(self):
        return {
            "metadata": self.metadata.to_dict(),
            "sections": [section.to_dict() for section in self.sections]
        }
    
    def model_dump(self):
        """Alias for to_dict for compatibility."""
        return self.to_dict()

print("✅ Schema definitions loaded")


In [ ]:
# Configuration Widgets
config_widgets = widgets.VBox([
    widgets.HTML("<h3>📋 OCR Configuration</h3>"),
    widgets.FloatSlider(
        value=0.0,
        min=0.0,
        max=2.0,
        step=0.1,
        description='Temperature:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    ),
    widgets.IntSlider(
        value=4096,
        min=512,
        max=8192,
        step=512,
        description='Max Tokens:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    ),
    widgets.Dropdown(
        options=['free_ocr', 'markdown', 'form', 'table'],
        value='free_ocr',
        description='Prompt Profile:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    ),
    widgets.Text(
        value='',
        placeholder='Optional: Document type hint (invoice, form, etc.)',
        description='Doc Type Hint:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
])

display(config_widgets)


In [ ]:
# Build Structured Prompt (Minimal - JSON Format Only)
def build_structured_prompt(prompt_profile="free_ocr", doc_type_hint="", image_size=None):
    """Build minimal prompt with only JSON format specification."""
    
    # Solid JSON schema example - this is the only thing we need
    schema_example = """{"document_type": "invoice", "sections": [{"section_id": "h1", "section_type": "header", "bbox": {"x1": 0, "y1": 0, "x2": 500, "y2": 100}, "key_value_pairs": [{"key": "Invoice", "value": "INV-123", "confidence": 0.95}], "text": "..."}]}"""
    
    # Minimal instruction - just the JSON format
    base_instruction = f"""Return JSON:\n{schema_example}"""
    
    prompt_mapping = {
        "free_ocr": base_instruction,
        "form": base_instruction,
        "table": base_instruction,
        "markdown": base_instruction
    }
    
    return prompt_mapping.get(prompt_profile, prompt_mapping["free_ocr"])


In [ ]:
# Bounding Box Conversion Function
def convert_bbox_to_pixels(bbox_data: dict, image_width: int, image_height: int) -> dict:
    """
    Convert bounding box coordinates to pixels.
    Handles both normalized [0, 1000) and pixel coordinates.
    
    Args:
        bbox_data: Dict with x1, y1, x2, y2 coordinates
        image_width: Image width in pixels
        image_height: Image height in pixels
    
    Returns:
        Dict with pixel coordinates
    """
    x1 = bbox_data.get("x1", 0)
    y1 = bbox_data.get("y1", 0)
    x2 = bbox_data.get("x2", 0)
    y2 = bbox_data.get("y2", 0)
    
    # Check if coordinates are normalized (typically < 1000)
    # If max coordinate is <= 1000, assume normalized format
    max_coord = max(abs(x1), abs(y1), abs(x2), abs(y2))
    
    if max_coord <= 1000 and max_coord > 0:
        # Normalized coordinates [0, 1000) - convert to pixels
        x1_pixel = int((x1 / 1000) * image_width)
        y1_pixel = int((y1 / 1000) * image_height)
        x2_pixel = int((x2 / 1000) * image_width)
        y2_pixel = int((y2 / 1000) * image_height)
        
        # Ensure coordinates are within image bounds
        x1_pixel = max(0, min(x1_pixel, image_width - 1))
        y1_pixel = max(0, min(y1_pixel, image_height - 1))
        x2_pixel = max(0, min(x2_pixel, image_width - 1))
        y2_pixel = max(0, min(y2_pixel, image_height - 1))
        
        # Ensure x2 >= x1 and y2 >= y1
        if x2_pixel < x1_pixel:
            x2_pixel = x1_pixel + 1
        if y2_pixel < y1_pixel:
            y2_pixel = y1_pixel + 1
        
        return {
            "x1": x1_pixel,
            "y1": y1_pixel,
            "x2": x2_pixel,
            "y2": y2_pixel
        }
    else:
        # Already in pixel coordinates - just ensure they're integers and within bounds
        x1_pixel = max(0, min(int(x1), image_width - 1))
        y1_pixel = max(0, min(int(y1), image_height - 1))
        x2_pixel = max(0, min(int(x2), image_width - 1))
        y2_pixel = max(0, min(int(y2), image_height - 1))
        
        # Ensure x2 >= x1 and y2 >= y1
        if x2_pixel < x1_pixel:
            x2_pixel = x1_pixel + 1
        if y2_pixel < y1_pixel:
            y2_pixel = y1_pixel + 1
        
        return {
            "x1": x1_pixel,
            "y1": y1_pixel,
            "x2": x2_pixel,
            "y2": y2_pixel
        }


In [ ]:
# Parse and Validate Structured Output
def parse_structured_output(raw_result: str, image_size: tuple, processing_time_ms: float) -> OCRResult:
    """
    Parse model output and validate against Pydantic schema.
    Handles bounding box coordinate conversion from normalized to pixels.
    """
    if not raw_result or not raw_result.strip():
        return OCRResult(
            metadata=DocumentMetadata(
                document_type="unknown",
                processing_time_ms=processing_time_ms,
                image_size={"width": image_size[0], "height": image_size[1]}
            ),
            sections=[]
        )
    
    text = raw_result.strip()
    json_text = text
    image_width, image_height = image_size
    
    # Extract JSON from markdown code blocks
    if "```json" in text:
        start = text.find("```json") + 7
        end = text.find("```", start)
        if end != -1:
            json_text = text[start:end].strip()
    elif "```" in text:
        start = text.find("```") + 3
        end = text.find("```", start)
        if end != -1:
            json_text = text[start:end].strip()
    
    # Try to find JSON object
    if "{" in json_text:
        start_idx = json_text.find("{")
        if start_idx != -1:
            # Find matching closing brace
            brace_count = 0
            end_idx = start_idx
            for i, char in enumerate(json_text[start_idx:], start_idx):
                if char == "{":
                    brace_count += 1
                elif char == "}":
                    brace_count -= 1
                    if brace_count == 0:
                        end_idx = i + 1
                        break
            json_text = json_text[start_idx:end_idx]
    
    try:
        # Parse JSON
        data = json.loads(json_text)
        
        # Ensure metadata exists
        if "metadata" not in data:
            data["metadata"] = {}
        
        # Set required metadata fields
        data["metadata"].update({
            "engine": "Qwen3-VL",
            "model_variant": "2B-Instruct",
            "processing_time_ms": processing_time_ms,
            "image_size": {"width": image_width, "height": image_height}
        })
        
        # Ensure document_type exists
        if "document_type" not in data["metadata"]:
            if "document_type" in data:
                data["metadata"]["document_type"] = data.pop("document_type")
            else:
                data["metadata"]["document_type"] = "unknown"
        
        # Ensure sections exist
        if "sections" not in data:
            data["sections"] = []
        
        # Process sections and convert bounding boxes
        processed_sections = []
        for section_data in data["sections"]:
            if not isinstance(section_data, dict):
                continue
            
            # Convert bounding box coordinates if present
            if "bbox" in section_data and section_data["bbox"]:
                bbox_data = section_data["bbox"]
                if isinstance(bbox_data, dict):
                    # Convert normalized coordinates to pixels
                    section_data["bbox"] = convert_bbox_to_pixels(
                        bbox_data, image_width, image_height
                    )
            
            processed_sections.append(section_data)
        
        data["sections"] = processed_sections
        
        # Validate with Pydantic
        result = OCRResult(**data)
        return result
        
    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing error: {e}")
        print(f"Raw output preview: {text[:500]}")
        return OCRResult(
            metadata=DocumentMetadata(
                document_type="unknown",
                processing_time_ms=processing_time_ms,
                image_size={"width": image_width, "height": image_height}
            ),
            sections=[]
        )
    except ValidationError as e:
        print(f"⚠️ Validation error: {e}")
        print(f"Attempting to fix structure...")
        # Try to create minimal valid structure
        try:
            doc_type = data.get("document_type", "unknown") if isinstance(data, dict) else "unknown"
            sections_data = data.get("sections", []) if isinstance(data, dict) else []
            
            # Try to validate sections individually with bbox conversion
            valid_sections = []
            for section_data in sections_data:
                try:
                    # Convert bbox if present
                    if "bbox" in section_data and section_data["bbox"]:
                        if isinstance(section_data["bbox"], dict):
                            section_data["bbox"] = convert_bbox_to_pixels(
                                section_data["bbox"], image_width, image_height
                            )
                    section = Section(**section_data)
                    valid_sections.append(section)
                except Exception as e:
                    print(f"⚠️ Skipping invalid section: {e}")
                    continue
            
            return OCRResult(
                metadata=DocumentMetadata(
                    document_type=doc_type,
                    processing_time_ms=processing_time_ms,
                    image_size={"width": image_width, "height": image_height}
                ),
                sections=valid_sections
            )
        except Exception as e:
            print(f"❌ Failed to create fallback structure: {e}")
            return OCRResult(
                metadata=DocumentMetadata(
                    document_type="unknown",
                    processing_time_ms=processing_time_ms,
                    image_size={"width": image_width, "height": image_height}
                ),
                sections=[]
            )


In [ ]:
# OCR Inference Function with Structured Output
def run_ocr_structured(image, prompt_profile="free_ocr", max_tokens=4096, 
                      temperature=0.0, doc_type_hint=""):
    """
    Run OCR inference with structured output.
    
    Args:
        image: PIL Image
        prompt_profile: One of "free_ocr", "markdown", "form", "table"
        max_tokens: Maximum tokens to generate
        temperature: Sampling temperature (0.0 for deterministic)
        doc_type_hint: Optional hint for document type
    
    Returns:
        OCRResult object with structured sections
    """
    # Convert image to RGB if needed
    if image.mode != "RGB":
        image = image.convert("RGB")
    
    # Save image to temporary file
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        image.save(tmp.name, format="PNG")
        tmp_path = tmp.name
    
    try:
        # Build structured prompt
        prompt = build_structured_prompt(prompt_profile, doc_type_hint, image.size)
        
        # Prepare messages
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": f"file://{tmp_path}"},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        
        # Process vision info
        image_inputs, video_inputs = process_vision_info(messages)
        
        # Apply chat template
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        
        # Process inputs
        inputs = processor(
            text=text,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        
        # Move inputs to device
        if hasattr(inputs, "to"):
            inputs = inputs.to(device)
        else:
            inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}
        
        # Generate
        print("🔄 Running inference...")
        start_time = time.time()
        with torch.no_grad():
            if temperature == 0.0:
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    do_sample=False,
                )
            else:
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    temperature=temperature,
                    do_sample=True,
                )
        
        # Extract generated tokens
        if isinstance(inputs, dict) and "input_ids" in inputs:
            input_ids = inputs["input_ids"]
            generated_ids_trimmed = [
                out_ids[len(in_ids):]
                for in_ids, out_ids in zip(input_ids, generated_ids)
            ]
        else:
            generated_ids_trimmed = generated_ids
        
        # Decode output
        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        
        inference_time = time.time() - start_time
        processing_time_ms = inference_time * 1000
        print(f"✅ Inference completed in {inference_time:.2f} seconds")
        
        raw_output = output_text[0] if output_text else ""
        
        # Parse and validate structured output
        result = parse_structured_output(raw_output, image.size, processing_time_ms)
        
        return result
    
    finally:
        # Clean up temp file
        Path(tmp_path).unlink(missing_ok=True)


In [ ]:
# Visualization function for sections
def visualize_sections(image, result: OCRResult):
    """
    Visualize OCR results with section-level bounding boxes.
    
    Args:
        image: PIL Image
        result: OCRResult object with sections
    """
    # Create a copy for drawing
    img_with_boxes = image.copy()
    draw = ImageDraw.Draw(img_with_boxes)
    
    # Color map for section types
    section_colors = {
        "header": "red",
        "body": "blue",
        "footer": "green",
        "table": "orange",
        "sidebar": "purple",
        "signature": "cyan",
        "logo": "magenta",
        "form_field": "yellow"
    }
    
    # Draw bounding boxes for each section
    for section in result.sections:
        bbox = section.bbox
        color = section_colors.get(section.section_type.lower(), "yellow")
        
        # Draw bounding box
        draw.rectangle(
            [bbox.x1, bbox.y1, bbox.x2, bbox.y2],
            outline=color,
            width=3
        )
        
        # Draw section label
        label = f"{section.section_type} ({section.section_id})"
        # Try to draw text above the box
        try:
            draw.text((bbox.x1, max(0, bbox.y1 - 20)), label, fill=color)
        except:
            # Fallback if font issues
            pass
    
    # Display image with boxes
    plt.figure(figsize=(15, 10))
    plt.imshow(img_with_boxes)
    plt.axis('off')
    plt.title(f"OCR Results - {len(result.sections)} sections detected | Document Type: {result.metadata.document_type}", 
               fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Display structured JSON output
    print("\n" + "="*80)
    print("📄 STRUCTURED JSON OUTPUT")
    print("="*80)
    print(json.dumps(result.model_dump(), indent=2))
    
    # Display key-value pairs summary
    print("\n" + "="*80)
    print("🔑 KEY-VALUE PAIRS SUMMARY")
    print("="*80)
    
    total_kv_pairs = sum(len(section.key_value_pairs) for section in result.sections)
    if total_kv_pairs == 0:
        print("No key-value pairs extracted.")
    else:
        for section in result.sections:
            if section.key_value_pairs:
                print(f"\n📌 Section: {section.section_type} ({section.section_id})")
                for kv in section.key_value_pairs:
                    print(f"   • {kv.key}: {kv.value} (confidence: {kv.confidence:.2f})")
    
    # Display sections summary
    print("\n" + "="*80)
    print("📋 SECTIONS SUMMARY")
    print("="*80)
    for i, section in enumerate(result.sections, 1):
        print(f"\n{i}. {section.section_type.upper()} - {section.section_id}")
        print(f"   BBox: ({section.bbox.x1}, {section.bbox.y1}) -> ({section.bbox.x2}, {section.bbox.y2})")
        print(f"   Key-Value Pairs: {len(section.key_value_pairs)}")
        print(f"   Text Preview: {section.text[:100]}..." if len(section.text) > 100 else f"   Text: {section.text}")


In [ ]:
# Upload Image and Run Structured OCR
print("📤 Upload an image file...")
uploaded = files.upload()

# Get the uploaded file
if uploaded:
    # Get widget values
    temperature = config_widgets.children[1].value
    max_tokens = config_widgets.children[2].value
    prompt_profile = config_widgets.children[3].value
    doc_type_hint = config_widgets.children[4].value.strip()
    
    # Get the first (and only) uploaded file
    file_name = list(uploaded.keys())[0]
    print(f"\n📄 Processing: {file_name}")
    
    # Load image
    image = Image.open(file_name)
    print(f"📐 Image size: {image.size}, Mode: {image.mode}")
    
    # Display configuration
    print(f"\n⚙️ Configuration:")
    print(f"   Temperature: {temperature}")
    print(f"   Max Tokens: {max_tokens}")
    print(f"   Prompt Profile: {prompt_profile}")
    if doc_type_hint:
        print(f"   Document Type Hint: {doc_type_hint}")
    
    # Run structured OCR
    print("\n" + "="*80)
    print("🚀 Running Structured OCR...")
    print("="*80)
    
    result = run_ocr_structured(
        image,
        prompt_profile=prompt_profile,
        max_tokens=max_tokens,
        temperature=temperature,
        doc_type_hint=doc_type_hint
    )
    
    print(f"\n✅ Found {len(result.sections)} sections")
    print(f"📊 Document Type: {result.metadata.document_type}")
    
    # Visualize results
    visualize_sections(image, result)
    
else:
    print("❌ No file uploaded!")


## Optional: Test with Image URL

You can also test with an image from a URL instead of uploading.


In [ ]:
# Optional: Test with image URL
import requests
from io import BytesIO

# Example: Test with an image URL
# image_url = "https://example.com/your-image.jpg"
# response = requests.get(image_url)
# image = Image.open(BytesIO(response.content))
# 
# # Run structured OCR
# result = run_ocr_structured(image, prompt_profile="free_ocr", doc_type_hint="invoice")
# visualize_sections(image, result)
